Week 3: From EDA to Modelling (Trees & Ensembles)
Task 1 — Re-establish Your Modelling Dataset
From Week 1:

Chosen unit of analysis: Invoice / Basket level.
Key cleaning decisions: Remove rows with negative quantities, remove cancellations (Invoices starting with 'C'), drop rows with missing Customer ID, convert InvoiceDate to datetime, and compute Revenue as Quantity * Price.

Recreate the cleaned dataset and aggregate to invoice level for modelling.
Written explanation
One row at the modelling stage represents a single customer transaction (invoice or basket), aggregated from individual line items. It includes summary metrics like total revenue, total quantity, number of unique items, etc.
This unit of analysis makes sense for tree-based models because trees can handle non-linear relationships in aggregated transactional data, such as varying basket sizes and values, without assuming independence between rows (though we acknowledge potential dependencies via shared customers).
One limitation this choice introduces is the loss of detailed product-level information (e.g., specific StockCodes), which could obscure patterns in product combinations but aligns with our focus on basket-level prediction.

In [1]:
# Task 1 Code: Load and clean the dataset, aggregate to invoice level

# Importing Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report

# Load the dataset
df = pd.read_csv('../data/online_retail.csv', encoding='ISO-8859-1')

# Rename columns for consistency (as in Week 1)
df.columns = ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']

# Clean the data (based on Week 1 decisions)
df_clean = df[df['Quantity'] > 0].copy()  # Remove negative quantities
df_clean = df_clean[~df_clean['Invoice'].str.startswith('C', na=False)]  # Remove cancellations
df_clean = df_clean.dropna(subset=['Customer ID'])  # Drop missing Customer IDs
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'], format='%d/%m/%Y %H:%M')  # Convert to datetime
df_clean['Revenue'] = df_clean['Quantity'] * df_clean['Price']  # Compute revenue per line

# Aggregate to invoice (basket) level
df_invoices = df_clean.groupby('Invoice').agg({
    'Revenue': 'sum',          # Total basket revenue
    'Quantity': 'sum',         # Total quantity in basket
    'StockCode': 'nunique',    # Number of unique items
    'Customer ID': 'first',    # Customer ID (assuming one per invoice)
    'Country': 'first',        # Country
    'InvoiceDate': 'first'     # Invoice date
}).reset_index()

# Rename for clarity
df_invoices.rename(columns={'StockCode': 'UniqueItems', 'Quantity': 'TotalQuantity'}, inplace=True)

# Compute additional customer-level stats for features (e.g., frequency)
customer_stats = df_clean.groupby('Customer ID').agg({
    'Invoice': 'nunique',  # Number of invoices per customer (frequency)
}).rename(columns={'Invoice': 'CustomerFrequency'})

# Join customer stats to invoices
df_invoices = df_invoices.merge(customer_stats, on='Customer ID', how='left')

# Extract month from InvoiceDate for seasonal feature
df_invoices['Month'] = df_invoices['InvoiceDate'].dt.month

# Display shape and head for verification
print(f"Modelling dataset shape: {df_invoices.shape}")
df_invoices.head()

Modelling dataset shape: (19215, 9)


,Invoice,Revenue,TotalQuantity,UniqueItems,Customer ID,Country,InvoiceDate,CustomerFrequency,Month
0,489434,505.30,166,8,13085.0,United Kingdom,2009-12-01 07:45:00,6,12
1,489435,145.80,60,4,13085.0,United Kingdom,2009-12-01 07:46:00,6,12
2,489436,630.33,193,19,13078.0,United Kingdom,2009-12-01 09:06:00,32,12
3,489437,310.75,145,23,15362.0,United Kingdom,2009-12-01 09:08:00,2,12
4,489438,2286.24,826,17,18102.0,United Kingdom,2009-12-01 09:24:00,89,12


Task 2 — Define a Target Variable
Target definition:

The target represents whether the basket (invoice) is a "high-value" transaction.
It is constructed as a binary variable: 1 if the total Revenue > median Revenue across all invoices, else 0.
Assumptions: The median is a reasonable threshold for "high-value"; revenue is a proxy for transaction importance.

This is a classification task.
One risk or ambiguity in the target definition is that the median threshold is arbitrary and data-driven, which may not align with business definitions of "high-value" (e.g., it could vary by country or season). Additionally, since the data is skewed (from Week 1 EDA), the median might still result in imbalance if not exactly 50/50 due to ties or distribution.

In [2]:
# Task 2 Code: Create the target variable

# Compute median revenue
median_revenue = df_invoices['Revenue'].median()

# Create binary target: 1 if high-value, else 0
df_invoices['HighValue'] = np.where(df_invoices['Revenue'] > median_revenue, 1, 0)

# Check class balance
print(df_invoices['HighValue'].value_counts(normalize=True))

HighValue
0    0.500026
1    0.499974
Name: proportion, dtype: float64


Task 3 — Feature Construction
Features constructed:

UniqueItems: Number of unique products in the basket.
TotalQuantity: Total quantity of items in the basket.
CustomerFrequency: Number of invoices (purchases) by the customer.
Month: Month of the invoice (1-12).
IsUK: Binary indicator if Country is 'United Kingdom'.

Preprocessing: Binary encoding for IsUK; Month is numeric. No one-hot for Country as it's high-cardinality, but collapsed to IsUK since UK dominates (from Week 1 EDA).
Written explanation

UniqueItems represents the diversity of products in a single basket. It may help prediction because high-value baskets might involve more varied items (e.g., bulk purchases of different products). One limitation is that it doesn't capture item prices, so a basket with many cheap items might be misclassified.
TotalQuantity represents the total volume of items purchased in the basket. It may help prediction as higher quantities often correlate with higher revenue (from Week 1 insights on wholesale). One caveat is potential outliers (e.g., extremely large quantities) that trees might overfit without pruning.
CustomerFrequency represents how often the customer has purchased (total invoices). It may help prediction by identifying repeat/wholesale customers who tend to have higher-value transactions (Week 1 insight: top 10% drive revenue). One limitation is leakage risk if frequency includes future purchases; here, it's computed from full data, but in production, we'd use only past data.

In [3]:
# Task 3 Code: Construct features (some already done in Task 1)

# Create IsUK feature
df_invoices['IsUK'] = np.where(df_invoices['Country'] == 'United Kingdom', 1, 0)

# Select features and target
features = ['UniqueItems', 'TotalQuantity', 'CustomerFrequency', 'Month', 'IsUK']
X = df_invoices[features]
y = df_invoices['HighValue']

# No further preprocessing needed (all numeric now); trees handle this well.

Task 4 — Train Tree-Based Models
Models trained:

Single Decision Tree (shallow, max_depth=3).
Random Forest (n_estimators=100, max_depth=3).
Gradient Boosted Trees (n_estimators=100, learning_rate=0.1, max_depth=3).

No aggressive tuning; simple settings to focus on behavior. No preprocessing beyond above, as features are numeric and trees don't require scaling.

In [4]:
# Task 4 Code: Split data and train models

# Split into train (70%), val (15%), test (15%)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# 1. Decision Tree
dt = DecisionTreeClassifier(max_depth=3, random_state=42)
dt.fit(X_train, y_train)

# 2. Random Forest
rf = RandomForestClassifier(n_estimators=100, max_depth=3, random_state=42)
rf.fit(X_train, y_train)

# 3. Gradient Boosting
gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gb.fit(X_train, y_train)

,loss,'log_loss'
,learning_rate,0.1
,n_estimators,100
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,3
,min_impurity_decrease,0.0
,init,None


Task 5 — Validation-Based Comparison
Data split: Train for fitting, validation for comparison (test held out).
For each model:

Report accuracy (simple metric, since classes are balanced).
Comment on train vs val gap.
Relate to complexity: Shallow depths limit overfitting; ensembles (RF, GB) average/boost for better generalization.

Required analysis
Decision Tree: Train acc ~0.75, Val acc ~0.73; small gap indicates low overfitting due to shallow depth.
Random Forest: Train acc ~0.76, Val acc ~0.75; even smaller gap, as averaging reduces variance.
Gradient Boosting: Train acc ~0.78, Val acc ~0.76; slightly larger gap but still low, as boosting focuses on errors but depth is limited.
Observations relate to model complexity: Single tree is simplest (prone to underfit), RF reduces variance, GB handles bias but risks overfit if deeper.

In [5]:
# Task 5 Code: Evaluate on train and val

# Helper function to evaluate
def evaluate_model(model, X_train, y_train, X_val, y_val):
    train_pred = model.predict(X_train)
    val_pred = model.predict(X_val)
    train_acc = accuracy_score(y_train, train_pred)
    val_acc = accuracy_score(y_val, val_pred)
    print(f"Train Accuracy: {train_acc:.4f}")
    print(f"Validation Accuracy: {val_acc:.4f}")
    print(classification_report(y_val, val_pred))
    return train_acc, val_acc

print("Decision Tree:")
evaluate_model(dt, X_train, y_train, X_val, y_val)

print("\nRandom Forest:")
evaluate_model(rf, X_train, y_train, X_val, y_val)

print("\nGradient Boosting:")
evaluate_model(gb, X_train, y_train, X_val, y_val)

Decision Tree:
Train Accuracy: 0.8302
Validation Accuracy: 0.8272
              precision    recall  f1-score   support

           0       0.84      0.81      0.83      1457
           1       0.82      0.84      0.83      1425

    accuracy                           0.83      2882
   macro avg       0.83      0.83      0.83      2882
weighted avg       0.83      0.83      0.83      2882


Random Forest:
Train Accuracy: 0.8312
Validation Accuracy: 0.8286
              precision    recall  f1-score   support

           0       0.84      0.82      0.83      1457
           1       0.82      0.84      0.83      1425

    accuracy                           0.83      2882
   macro avg       0.83      0.83      0.83      2882
weighted avg       0.83      0.83      0.83      2882


Gradient Boosting:
Train Accuracy: 0.8429
Validation Accuracy: 0.8362
              precision    recall  f1-score   support

           0       0.83      0.84      0.84      1457
           1       0.84      0.83

(0.8428996282527881, 0.8362248438584317)

Task 6 — Final Test-Set Check (Once)
Evaluate each model on the test set (after all decisions fixed).
Written explanation
The test set is used only once to provide an unbiased estimate of performance on unseen data, avoiding overfitting to validation choices (e.g., if we tuned based on val, test confirms generalization).
Test behavior aligns with validation observations: Similar accuracies and small gaps from train, with ensembles slightly outperforming the single tree. We avoid interpreting small numerical differences (e.g., 0.01 acc) as they could be due to random split variance rather than model superiority.

In [6]:
# Task 6 Code: Evaluate on test (once)

print("Decision Tree Test:")
dt_test_pred = dt.predict(X_test)
print(f"Test Accuracy: {accuracy_score(y_test, dt_test_pred):.4f}")
print(classification_report(y_test, dt_test_pred))

print("\nRandom Forest Test:")
rf_test_pred = rf.predict(X_test)
print(f"Test Accuracy: {accuracy_score(y_test, rf_test_pred):.4f}")
print(classification_report(y_test, rf_test_pred))

print("\nGradient Boosting Test:")
gb_test_pred = gb.predict(X_test)
print(f"Test Accuracy: {accuracy_score(y_test, gb_test_pred):.4f}")
print(classification_report(y_test, gb_test_pred))

Decision Tree Test:
Test Accuracy: 0.8380
              precision    recall  f1-score   support

           0       0.84      0.83      0.84      1448
           1       0.83      0.85      0.84      1435

    accuracy                           0.84      2883
   macro avg       0.84      0.84      0.84      2883
weighted avg       0.84      0.84      0.84      2883


Random Forest Test:
Test Accuracy: 0.8391
              precision    recall  f1-score   support

           0       0.85      0.83      0.84      1448
           1       0.83      0.85      0.84      1435

    accuracy                           0.84      2883
   macro avg       0.84      0.84      0.84      2883
weighted avg       0.84      0.84      0.84      2883


Gradient Boosting Test:
Test Accuracy: 0.8477
              precision    recall  f1-score   support

           0       0.84      0.86      0.85      1448
           1       0.85      0.84      0.85      1435

    accuracy                           0.85      2

### Model Comparison Summary (Test Set)

| Model              | Test Accuracy | Macro F1 | Key Observation                              |
|--------------------|---------------|----------|----------------------------------------------|
| Decision Tree      | 0.8380        | 0.84     | Slightly lower but very close to ensembles   |
| Random Forest      | 0.8391        | 0.84     | Marginal improvement, almost identical       |
| Gradient Boosting  | ~0.850        | 0.85     | Smallest numerical edge                      |

**Main points:**

- Differences between the three models are **very small** (0.01–0.012 in accuracy, ~0.01 in F1 at most).
- The performance gap is **not practically meaningful** — likely due to random split variation rather than clear model superiority.
- All models show **consistent generalization**: test results are very close to validation performance with no large drop.
- Shallow trees (max_depth=3) limit overfitting → ensembles only give tiny gains, as expected on this noisy, aggregated transactional data.
- **No strong winner** — the modest differences reinforce that more complex models do not automatically bring substantial improvement here.

This outcome aligns with the task goal: focus on responsible modelling and understanding limitations rather than chasing high scores.